# 太陽フィラメント分割：Kaggle実習

日本語プレゼンテーションの補助Notebookです。Kaggleへこの `.ipynb` をインポートし、大会データをInputへ追加して、GPUとInternetを確認してください。

**教育用の未学習ベースラインです。ここではGPU学習・Kaggle提出を実行していません。**

- `METHOD` で `yolo_seg`、`yolo_sam`、`unet` を選び、1方式ずつ実行します。
- 最初は `SMOKE=True` で全工程を小さく通します。`submission_SMOKE_DO_NOT_SUBMIT.csv` は一部画像のみで、提出用ではありません。
- 本実験は `SMOKE=False` にして、新しい出力フォルダで上から実行します。
- 1クラス分割、元画像単位の固定分割、各注釈者を独立に扱う評価を使います。
- 512入力は処理確認用です。高解像度やタイル、高度な個体分離は別の実験として比較します。

大会の規約と最新の自己評価コードを優先してください。外部のテスト正解を取得・照合しません。

## 1. ライブラリ

PyTorchはKaggle環境のものをまず使います。導入後にKernel再起動が必要な場合は、このセルの次から実行します。Internetを使えない場合は対応するライブラリと重みをInputとして準備します。

In [ ]:
%pip install -q ultralytics pycocotools
%pip install -q segmentation-models-pytorch

## 2. 実験設定

`batch` は一度に計算する枚数、`epochs` は学習データの最大周回数、`lr` は重みの更新幅です。値は出発点の例です。最初は変更を少なくして、原因と結果の関係を確認します。2GPU拡張はプレゼンテーションの付録を参照してください。このNotebookの基本経路は1GPUです。

In [ ]:
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
import os, json, random, hashlib, shutil, time, itertools
import importlib.metadata as metadata
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pycocotools.coco import COCO
from pycocotools import mask as mask_utils

METHOD = 'yolo_seg'  # 'yolo_seg', 'yolo_sam', 'unet'
SMOKE = True
SEED = 42
IMAGE_SIZE = 512
BATCH = 4
WORKERS = min(2, os.cpu_count() or 1)
EPOCHS = 1 if SMOKE else 30
LR = 3e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 7  # U-NetではPQを測った回数を基準に待つ
PQ_EVERY = 1 if SMOKE else 2
VAL_FRACTION = 0.2
RESUME_UNET = ''  # 自分で保存したlast_unet.pthのパスを指定できる
RUN_ID = f'{METHOD}_{"smoke" if SMOKE else "full"}_{int(time.time())}'
OUT = Path('/kaggle/working') / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)
assert METHOD in {'yolo_seg', 'yolo_sam', 'unet'}
assert IMAGE_SIZE % 32 == 0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
DEVICE_ARG = 0 if DEVICE.type == 'cuda' else 'cpu'
AMP = DEVICE.type == 'cuda'
versions = {name: metadata.version(name) for name in
    ['torch', 'ultralytics', 'segmentation-models-pytorch', 'pycocotools']}
config = dict(method=METHOD, smoke=SMOKE, seed=SEED, image_size=IMAGE_SIZE,
              batch=BATCH, workers=WORKERS, epochs=EPOCHS, lr=LR,
              weight_decay=WEIGHT_DECAY, patience=PATIENCE,
              pq_every=PQ_EVERY, val_fraction=VAL_FRACTION,
              versions=versions, device=str(DEVICE))
(OUT / 'config.json').write_text(json.dumps(config, indent=2))
print(config)
print('GPU count:', torch.cuda.device_count())
if AMP:
    print(torch.cuda.get_device_name(0))

## 3. データの場所とCOCOを確認

`Path` はパスを扱い、`rglob` は下位フォルダも探します。`assert` は想定と違う場合に止めます。複数の候補がある場合、表示を確認して使うものを明示的に選んでください。

In [ ]:
ANN_NAME = 'MAGFiLO_1.0_Annotations_kaggle2026_train.json'
found = sorted(Path('/kaggle/input').rglob(ANN_NAME))
assert len(found) == 1, f'Annotation candidates: {found}'
ANN_PATH = found[0]
ROOT = ANN_PATH.parent.parent
TRAIN_DIR = ROOT / 'train' / 'train_images'
TEST_DIR = ROOT / 'test' / 'test_images'
assert TRAIN_DIR.is_dir() and TEST_DIR.is_dir()
coco = COCO(str(ANN_PATH))
records = sorted(coco.loadImgs(coco.getImgIds()), key=lambda r: str(r['id']))
assert len(records) == len({str(r['id']) for r in records})
assert records, 'No training records'

def annotations(r):
    return coco.loadAnns(coco.getAnnIds(imgIds=[r['id']]))

def grayscale(path):
    with Image.open(path) as im:
        return np.array(im.convert('L'), dtype=np.uint8)

for r in records:
    assert (TRAIN_DIR / r['file_name']).is_file(), r['file_name']
    assert (r['height'], r['width']) == (2048, 2048)
    assert not any(a.get('iscrowd', 0) for a in annotations(r)), 'Crowd handling needed'
test_paths = sorted(p for p in TEST_DIR.iterdir()
    if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
assert test_paths and len({p.stem for p in test_paths}) == len(test_paths)
print('JSON:', ANN_PATH)
print('annotation records:', len(records), 'test images:', len(test_paths))
print('first record:', records[0])

## 4. 同じ元画像を同じ側へまとめる

画像を小さく切り出す前に分割します。ファイル名だけでなく、デコードした画素が完全一致する画像も同じグループへ置きます。**近い時刻の似た観測はハッシュだけでは防げないため、発展実験では観測日・時刻による分割も検討します。**

学習はグループごとに最初の注釈レコードを1つ選ぶ単純な方針です。検証は各注釈レコードを独立に保持します。複数注釈の不一致を扱う高度な学習法は別の実験です。

In [ ]:
names = sorted({r['file_name'] for r in records})
def image_hash(name):
    pixels = grayscale(TRAIN_DIR / name)
    assert pixels.shape == (2048, 2048), name
    return name, hashlib.sha256(pixels.tobytes()).hexdigest()

with ThreadPoolExecutor(max_workers=max(1, WORKERS)) as pool:
    source_hash = dict(pool.map(image_hash, names))
groups = sorted(set(source_hash.values()))
assert len(groups) >= 2
rng = np.random.default_rng(SEED)
rng.shuffle(groups)
n_val = min(len(groups) - 1, max(1, round(len(groups) * VAL_FRACTION)))
val_groups = set(groups[:n_val])
train_groups = set(groups[n_val:])
if SMOKE:
    train_groups = set(sorted(train_groups)[:8])
    val_groups = set(sorted(val_groups)[:2])

train_records_all = [r for r in records if source_hash[r['file_name']] in train_groups]
val_records = [r for r in records if source_hash[r['file_name']] in val_groups]
train_choice = {}
for r in train_records_all:
    train_choice.setdefault(source_hash[r['file_name']], r)
train_records = list(train_choice.values())
val_choice = {}
for r in val_records:
    val_choice.setdefault(r['file_name'], r)
val_selected = list(val_choice.values())
assert train_records and val_records
assert train_groups.isdisjoint(val_groups)
assert not ({r['file_name'] for r in train_records} & set(val_choice))
split = dict(seed=SEED, policy='decoded-pixel-hash, first-record-per-train-source',
    train_records=[r['id'] for r in train_records],
    validation_records=[r['id'] for r in val_records],
    source_hashes=source_hash)
(OUT / 'split.json').write_text(json.dumps(split, indent=2))
print('train sources:', len(train_records))
print('validation files:', len(val_choice), 'annotation records:', len(val_records))

## 5. 画像と正解の重ね合わせ

学習前に対応が正しいか目で確認します。ここで表示するのは学習に利用可能な画像の注釈です。マスクを合体して評価することはしません。

In [ ]:
r = train_records[0]
pixels = grayscale(TRAIN_DIR / r['file_name'])
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(pixels, cmap='gray')
axes[0].set_title('Input')
axes[1].imshow(pixels, cmap='gray')
for a in annotations(r):
    m = coco.annToMask(a)
    if m.any():
        axes[1].contour(m, levels=[0.5], linewidths=0.6)
axes[1].set_title('Independent annotation masks')
for ax in axes: ax.axis('off')
plt.show()
print('shape, dtype, min, max:', pixels.shape, pixels.dtype, pixels.min(), pixels.max())

## 6. RLEと主催者方式のPQ

PQの対応は **IoU > 0.5**。提供自己評価Notebookに合わせ、全該当ペアを数え、Hungarian割当はしません。重なり可能な独立マスクでは一般の一対一割当と差が出る場合があります。注釈レコード全体の分子とTP/FP/FNを合算し、最後に割ります。

COCOのRLE演算でIoUを求めるため、全マスクの大きなfloat配列を同時に持つ必要を減らせます。

In [ ]:
def mask_rle(mask):
    m = np.asarray(mask)
    assert m.shape == (2048, 2048)
    assert np.isin(m, [0, 1]).all()
    return mask_utils.encode(np.asfortranarray(m, dtype=np.uint8))

def counts_string(mask):
    return mask_rle(mask)['counts'].decode('ascii')

gt_rles = {str(r['id']): [mask_rle(coco.annToMask(a)) for a in annotations(r)]
           for r in val_records}

def score_rles(gt, pred):
    ng, npred = len(gt), len(pred)
    if ng == 0 or npred == 0:
        return dict(iou_sum=0.0, tp=0, fp=npred, fn=ng,
                    split_gt=0, merged_pred=0)
    # COCO returns prediction rows, GT columns. Transpose to GT x prediction.
    iou = np.asarray(mask_utils.iou(pred, gt, [0] * ng), dtype=float).T
    hit = iou > 0.5
    positive = iou > 0
    return dict(iou_sum=float(iou[hit].sum()), tp=int(hit.sum()),
                fp=int((hit.sum(axis=0) == 0).sum()),
                fn=int((hit.sum(axis=1) == 0).sum()),
                split_gt=int((positive.sum(axis=1) > 1).sum()),
                merged_pred=int((positive.sum(axis=0) > 1).sum()))

def evaluate_pq(settings, predictor=None):
    predictor = predictor or predict_instances
    # Each physical filename is inferred once, then compared with each annotation record.
    predictions = {}
    for name in sorted(val_choice):
        masks = predictor(TRAIN_DIR / name, settings)
        predictions[name] = [mask_rle(m) for m in masks]
    totals = dict(iou_sum=0.0, tp=0, fp=0, fn=0, split_gt=0, merged_pred=0)
    per_record = []
    for r in val_records:
        value = score_rles(gt_rles[str(r['id'])], predictions[r['file_name']])
        per_record.append(dict(record_id=str(r['id']), **value))
        for k in totals: totals[k] += value[k]
    denom = totals['tp'] + 0.5 * totals['fp'] + 0.5 * totals['fn']
    totals['pq'] = totals['iou_sum'] / denom if denom > 0 else 0.0
    return totals, pd.DataFrame(per_record)

# Logical check: a 10-pixel object split into two equal masks has union Dice 1,
# but each instance IoU is 0.5, which does not meet the strict PQ threshold.
gt_toy = np.zeros((2048, 2048), np.uint8)
gt_toy[100, 100:110] = 1
p1, p2 = np.zeros_like(gt_toy), np.zeros_like(gt_toy)
p1[100, 100:105] = 1
p2[100, 105:110] = 1
toy = score_rles([mask_rle(gt_toy)], [mask_rle(p1), mask_rle(p2)])
assert toy['tp'] == 0 and toy['fp'] == 2 and toy['fn'] == 1
assert np.array_equal(mask_utils.decode(mask_rle(gt_toy)), gt_toy)
print('RLE round trip and strict-IoU example passed.')

## 7. YOLO用データセットの作成

`yolo_seg` は輪郭の点列、`yolo_sam` は検出枠のラベルを作ります。配布データの1個体1ポリゴンを前提とする例です。想定外の複数輪郭やRLEを黙って間引かず、対応処理が必要な場合は停止します。画像と同名の `.txt` を作り、クラスは全て0へ統合します。

注釈者別GTは前のセルに独立して残してあり、YOLO標準validationの結果と大会PQを区別します。

In [ ]:
DATA_DIR = OUT / ('yolo_seg' if METHOD == 'yolo_seg' else 'yolo_detect')
YAML_PATH = DATA_DIR / 'dataset.yaml'
if METHOD in {'yolo_seg', 'yolo_sam'}:
    for split_name, chosen in [('train', train_records), ('val', val_selected)]:
        image_dir = DATA_DIR / 'images' / split_name
        label_dir = DATA_DIR / 'labels' / split_name
        image_dir.mkdir(parents=True, exist_ok=True)
        label_dir.mkdir(parents=True, exist_ok=True)
        assert len({Path(r['file_name']).stem for r in chosen}) == len(chosen)
        for r in chosen:
            source = TRAIN_DIR / r['file_name']
            shutil.copy2(source, image_dir / source.name)
            W, H = r['width'], r['height']
            lines = []
            for a in annotations(r):
                if METHOD == 'yolo_seg':
                    parts = a['segmentation']
                    if not isinstance(parts, list) or len(parts) != 1:
                        raise ValueError('Explicit multipart/RLE conversion required')
                    xy = np.asarray(parts[0], float).reshape(-1, 2)
                    assert len(xy) >= 3
                    xy /= np.array([W, H])
                    assert np.isfinite(xy).all() and ((xy >= 0) & (xy <= 1)).all()
                    values = xy.ravel()
                else:
                    x, y, w, h = a['bbox']
                    values = np.array([(x + w/2)/W, (y + h/2)/H, w/W, h/H])
                    assert ((values >= 0) & (values <= 1)).all()
                lines.append('0 ' + ' '.join(f'{v:.6f}' for v in values))
            (label_dir / f'{source.stem}.txt').write_text('\n'.join(lines))
    yaml_text = (f'path: {DATA_DIR}\ntrain: images/train\nval: images/val\n'
                 'names:\n  0: filament\n')
    YAML_PATH.write_text(yaml_text)
    print(YAML_PATH.read_text())
else:
    print('U-Net uses COCO masks directly; YOLO export skipped.')

## 8. YOLOの学習（YOLO方式を選んだ場合のみ）

`-seg.pt` は個体別マスク、`.pt` の検出器はYOLO＋SAM用の枠を学習します。小さいnモデルを初期例にします。標準の `best.pt` はUltralytics内部のfitnessで選択されます。**大会PQで最良のepochと同じとは限りません。** 下ではbestとlastを改めてPQで比較できるようにします。

In [ ]:
yolo = None
sam_model = None
YOLO_CANDIDATES = []
if METHOD in {'yolo_seg', 'yolo_sam'}:
    from ultralytics import YOLO, SAM
    initial = 'yolo26n-seg.pt' if METHOD == 'yolo_seg' else 'yolo26n.pt'
    yolo = YOLO(initial)
    args = dict(data=str(YAML_PATH), imgsz=IMAGE_SIZE, epochs=EPOCHS,
        batch=BATCH, lr0=LR, optimizer='AdamW', weight_decay=WEIGHT_DECAY,
        patience=PATIENCE, workers=WORKERS, device=DEVICE_ARG,
        seed=SEED, amp=AMP, hsv_h=0.0, hsv_s=0.0, hsv_v=0.05,
        mosaic=0.0, mixup=0.0, project=str(OUT), name='train')
    if METHOD == 'yolo_seg':
        args.update(mask_ratio=2, overlap_mask=False)
    yolo.train(**args)
    weights_dir = Path(yolo.trainer.save_dir) / 'weights'
    YOLO_CANDIDATES = [p for p in [weights_dir/'best.pt', weights_dir/'last.pt']
                       if p.is_file()]
    assert YOLO_CANDIDATES
    yolo = YOLO(str(YOLO_CANDIDATES[0]))
    if METHOD == 'yolo_sam':
        sam_model = SAM('sam2.1_t.pt')
        print('SAM weights stay fixed. Only the detector was trained.')
else:
    print('YOLO training skipped.')

## 9. U-NetのDatasetと損失

同じ注釈レコード内の前景を合体して二値学習します。評価には合体前の個体マスクを使います。入力は0〜1の1チャンネルです。画像とマスクには同じ反転・90度回転を加え、マスクのリサイズは最近傍補間を使います。

In [ ]:
class SolarDataset(Dataset):
    def __init__(self, records, augment=False):
        self.records, self.augment = records, augment
    def __len__(self): return len(self.records)
    def __getitem__(self, index):
        r = self.records[index]
        with Image.open(TRAIN_DIR / r['file_name']) as image:
            image = image.convert('L').resize((IMAGE_SIZE, IMAGE_SIZE),
                Image.Resampling.BILINEAR)
        mask = np.zeros((r['height'], r['width']), np.uint8)
        for a in annotations(r): mask |= coco.annToMask(a)
        mask = Image.fromarray(mask).resize((IMAGE_SIZE, IMAGE_SIZE),
            Image.Resampling.NEAREST)
        x, y = np.array(image, np.float32)/255, np.array(mask, np.float32)
        if self.augment:
            if random.random() < 0.5: x, y = x[:, ::-1], y[:, ::-1]
            if random.random() < 0.5: x, y = x[::-1], y[::-1]
            k = random.randrange(4)
            x, y = np.rot90(x, k), np.rot90(y, k)
        return (torch.from_numpy(np.ascontiguousarray(x[None])),
                torch.from_numpy(np.ascontiguousarray(y[None])))

def loss_fn(logits, target):
    bce = F.binary_cross_entropy_with_logits(logits, target)
    p = logits.sigmoid()
    axes = (1, 2, 3)
    dice = (2*(p*target).sum(axes) + 1e-6) / (
        p.sum(axes) + target.sum(axes) + 1e-6)
    return 0.5*bce + 0.5*(1-dice).mean()

def seed_worker(worker_id):
    value = torch.initial_seed() % 2**32
    np.random.seed(value)
    random.seed(value)

net = None
if METHOD == 'unet':
    import segmentation_models_pytorch as smp
    net = smp.Unet(encoder_name='resnet18', encoder_weights='imagenet',
                   in_channels=1, classes=1, activation=None).to(DEVICE)
    generator = torch.Generator().manual_seed(SEED)
    loader_args = dict(batch_size=BATCH, num_workers=WORKERS,
                       pin_memory=AMP, worker_init_fn=seed_worker)
    if WORKERS > 0: loader_args.update(persistent_workers=True, prefetch_factor=2)
    train_loader = DataLoader(SolarDataset(train_records, True), shuffle=True,
                              generator=generator, **loader_args)
    val_loader = DataLoader(SolarDataset(val_selected), shuffle=False, **loader_args)
    x, y = next(iter(train_loader))
    assert x.shape == y.shape
    print('input / target:', x.shape, y.shape)

## 10. 共通の推論関数

全方式が2048×2048の独立boolマスクのリストを返します。U-Netは確率を元解像度へ補間してから二値化し、8近傍の連結成分へ分けます。YOLOとSAMはAPIの座標復元後のサイズを検査します。**SAMへは元画像上のxyxy枠を渡します。**

`min_area` は元解像度での面積です。小さい本物まで消さないかvalidationで比較します。YOLOのAPI出力は既に二値マスクの場合があるため、生の確率閾値を調整しているとは見なしません。

In [ ]:
def filter_masks(masks, min_area):
    answer = []
    for m in masks:
        m = np.asarray(m, dtype=bool)
        assert m.shape == (2048, 2048), f'Unexpected mask shape: {m.shape}'
        if int(m.sum()) >= min_area and m.any(): answer.append(m)
    return answer

def components(probability, threshold, min_area):
    b = (probability >= threshold).astype(np.uint8)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(b, connectivity=8)
    return [labels == k for k in range(1, n)
            if stats[k, cv2.CC_STAT_AREA] >= min_area]

def unet_probability(path):
    with Image.open(path) as image:
        W, H = image.size
        assert (W, H) == (2048, 2048)
        small = image.convert('L').resize((IMAGE_SIZE, IMAGE_SIZE),
                                         Image.Resampling.BILINEAR)
        x = np.array(small, np.float32) / 255
    x = torch.from_numpy(x[None, None]).to(DEVICE)
    net.eval()
    with torch.inference_mode():
        probability = net(x).sigmoid()
        probability = F.interpolate(probability, size=(H, W),
            mode='bilinear', align_corners=False)
    return probability[0, 0].cpu().numpy()

def predict_instances(path, settings):
    if METHOD == 'unet':
        return components(unet_probability(path), settings['mask_thr'],
                          settings['min_area'])
    result = yolo.predict(str(path), imgsz=IMAGE_SIZE,
        conf=settings['conf'], iou=settings.get('nms_iou', 0.7),
        retina_masks=True, device=DEVICE_ARG, verbose=False)[0]
    if METHOD == 'yolo_seg':
        masks = [] if result.masks is None else result.masks.data.cpu().numpy() > 0.5
    else:
        boxes = result.boxes.xyxy.cpu().numpy().copy()
        if len(boxes) == 0: return []
        with Image.open(path) as image: W, H = image.size
        pad = settings.get('box_pad', 0.0)
        dx = (boxes[:, 2] - boxes[:, 0]) * pad
        dy = (boxes[:, 3] - boxes[:, 1]) * pad
        boxes[:, 0] -= dx; boxes[:, 2] += dx
        boxes[:, 1] -= dy; boxes[:, 3] += dy
        boxes[:, [0, 2]] = boxes[:, [0, 2]].clip(0, W)
        boxes[:, [1, 3]] = boxes[:, [1, 3]].clip(0, H)
        result = sam_model(str(path), bboxes=boxes.tolist(),
                           device=DEVICE_ARG, verbose=False)[0]
        masks = [] if result.masks is None else result.masks.data.cpu().numpy() > 0.5
    return filter_masks(masks, settings['min_area'])

BASE_SETTINGS = dict(mask_thr=0.5, conf=0.25, nms_iou=0.7,
                     box_pad=0.0, min_area=20)

## 11. U-Netの学習・検証・保存

各epochで学習と検証lossを測ります。PQは最初のepochと`PQ_EVERY`回ごとに測り、最良を保存します。`patience` はこの例ではPQを測定した回数です。全体の所要時間に検証も含めます。

`best_unet.pth` は推論用の重み、`last_unet.pth` はoptimizer等を含む再開用の状態です。中断・再開はepoch境界での再開であり、DataLoaderのprefetch等までビット単位の一致を保証するものではありません。

In [ ]:
history = []
if METHOD == 'unet':
    optimizer = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,
        mode='max', factor=0.5, patience=2)
    best_score, wait, start_epoch = -1.0, 0, 0
    if RESUME_UNET:
        # Load only a checkpoint produced by your own trusted run.
        state = torch.load(RESUME_UNET, map_location=DEVICE, weights_only=False)
        assert state['config']['image_size'] == IMAGE_SIZE
        assert state['split']['train_records'] == split['train_records']
        assert state['split']['validation_records'] == split['validation_records']
        net.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        scaler.load_state_dict(state['scaler'])
        scheduler.load_state_dict(state['scheduler'])
        start_epoch, best_score, wait = state['epoch'] + 1, state['best_score'], state['wait']
        history = state['history']
        random.setstate(state['random_state'])
        np.random.set_state(state['numpy_state'])
        torch.set_rng_state(state['torch_state'].cpu())
        generator.set_state(state['loader_generator_state'].cpu())
        if AMP and state['cuda_state'] is not None:
            torch.cuda.set_rng_state_all([v.cpu() for v in state['cuda_state']])
        previous_best = Path(RESUME_UNET).parent / 'best_unet.pth'
        assert previous_best.is_file(), 'Resume also needs the saved best weights'
        shutil.copy2(previous_best, OUT / 'best_unet.pth')

    for epoch in range(start_epoch, EPOCHS):
        t0 = time.perf_counter()
        net.train()
        train_sum = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=AMP):
                loss = loss_fn(net(x), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_sum += float(loss.detach()) * len(x)
        net.eval()
        val_sum = 0.0
        with torch.inference_mode():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                val_sum += float(loss_fn(net(x), y)) * len(x)
        score = None
        if epoch == 0 or (epoch + 1) % PQ_EVERY == 0 or epoch == EPOCHS - 1:
            score, _ = evaluate_pq(BASE_SETTINGS)
            scheduler.step(score['pq'])
            if score['pq'] > best_score:
                best_score, wait = score['pq'], 0
                torch.save(net.state_dict(), OUT / 'best_unet.pth')
            else: wait += 1
        row = dict(epoch=epoch+1, train_loss=train_sum/len(train_loader.dataset),
            val_loss=val_sum/len(val_loader.dataset), pq=None if score is None else score['pq'],
            lr=optimizer.param_groups[0]['lr'], seconds=time.perf_counter()-t0)
        history.append(row)
        pd.DataFrame(history).to_csv(OUT / 'history.csv', index=False)
        torch.save(dict(model=net.state_dict(), optimizer=optimizer.state_dict(),
            scheduler=scheduler.state_dict(), scaler=scaler.state_dict(), epoch=epoch,
            best_score=best_score, wait=wait, config=config, split=split, history=history,
            random_state=random.getstate(), numpy_state=np.random.get_state(),
            torch_state=torch.get_rng_state(), loader_generator_state=generator.get_state(),
            cuda_state=torch.cuda.get_rng_state_all() if AMP else None), OUT/'last_unet.pth')
        print(row)
        if wait >= PATIENCE: break
    net.load_state_dict(torch.load(OUT/'best_unet.pth', map_location=DEVICE, weights_only=True))
else:
    print('U-Net training skipped.')

## 12. 検証データだけで設定を選ぶ

重みと同じvalidation集合で閾値を比較します。U-Netは画素閾値、YOLOはconfidence、YOLO＋SAMはconfidenceと枠の余白を比較します。YOLOでは内部bestとlastも比較します。これは全epochのPQを調べた探索とは異なります。

実験を増やすとvalidationへも過適合します。最後に別の時期の観測等で再確認する設計も検討してください。ここから先、testの隠された正解は利用しません。

In [ ]:
if METHOD == 'unet':
    grid = [dict(BASE_SETTINGS, mask_thr=t) for t in ([0.5] if SMOKE else [0.35, 0.5, 0.65])]
    checkpoints = [None]
elif METHOD == 'yolo_seg':
    grid = [dict(BASE_SETTINGS, conf=c) for c in ([0.25] if SMOKE else [0.15, 0.25, 0.4])]
    checkpoints = YOLO_CANDIDATES[:1] if SMOKE else YOLO_CANDIDATES
else:
    grid = [dict(BASE_SETTINGS, conf=c, box_pad=p)
            for c, p in itertools.product([0.25] if SMOKE else [0.15, 0.3],
                                           [0.0] if SMOKE else [0.0, 0.05])]
    checkpoints = YOLO_CANDIDATES[:1] if SMOKE else YOLO_CANDIDATES
trials, best_trial = [], None
for checkpoint in checkpoints:
    if checkpoint is not None: yolo = YOLO(str(checkpoint))
    for settings in grid:
        t0 = time.perf_counter()
        metrics, details = evaluate_pq(settings)
        trial = dict(checkpoint=None if checkpoint is None else str(checkpoint),
                     settings=settings.copy(), metrics=metrics,
                     seconds=time.perf_counter()-t0)
        trials.append(trial)
        print(trial)
        if best_trial is None or metrics['pq'] > best_trial['metrics']['pq']:
            best_trial = trial
            details.to_csv(OUT/'best_validation_records.csv', index=False)
settings = best_trial['settings']
if best_trial['checkpoint'] is not None: yolo = YOLO(best_trial['checkpoint'])
(OUT/'validation_trials.json').write_text(json.dumps(trials, indent=2))
(OUT/'selected_settings.json').write_text(json.dumps(best_trial, indent=2))
print('Selected:', best_trial)

## 13. 見逃し・余分な予測・分断・結合を画像で確認

良い例だけを選ばず、小さい・長い・薄い・近接した物体のある画像も確認します。ここでは数枚を例示します。同じ検証画像と同じ表示条件で3方式を比べてください。

In [ ]:
for r in val_selected[:3]:
    path = TRAIN_DIR / r['file_name']
    prediction = predict_instances(path, settings)
    pixels = grayscale(path)
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax in axes: ax.imshow(pixels, cmap='gray'); ax.axis('off')
    for a in annotations(r):
        m = coco.annToMask(a)
        if m.any(): axes[1].contour(m, levels=[0.5], linewidths=0.6)
    for m in prediction:
        axes[2].contour(m, levels=[0.5], linewidths=0.6)
    axes[0].set_title('Input')
    axes[1].set_title('GT: one annotation record')
    axes[2].set_title(f'Prediction: {len(prediction)} instances')
    plt.show()

## 14. 全テスト推論とCSV

採用設定を固定してから実行します。0本の画像は行を作らず、完了記録へ残します。RLEは2048×2048の元画像座標で作り、`counts`の文字列だけをCSVへ入れます。

**SMOKE=TrueのCSVは一部画像だけです。提出しないでください。** 本実験の出力は`submission.csv`です。

In [ ]:
paths_to_run = test_paths[:3] if SMOKE else test_paths
rows, completed = [], []
journal = OUT / 'test_predictions'
journal.mkdir(exist_ok=True)
for path in paths_to_run:
    t0 = time.perf_counter()
    with Image.open(path) as image: assert image.size == (2048, 2048)
    masks = predict_instances(path, settings)
    image_rows = []
    for k, m in enumerate(masks, 1):
        if not m.any(): continue
        counts = counts_string(m)
        decoded = mask_utils.decode(dict(size=[2048, 2048], counts=counts.encode('ascii')))
        assert np.array_equal(decoded, np.asarray(m, np.uint8))
        image_rows.append(dict(filament_id=f'{path.stem}_{k}', segmentation_rle=counts))
    # Image-level journal preserves completed work without keeping dense masks in RAM.
    (journal/f'{path.stem}.json').write_text(json.dumps(image_rows))
    rows.extend(image_rows)
    completed.append(dict(file_name=path.name, instances=len(image_rows),
                          seconds=time.perf_counter()-t0))
    print(path.name, len(image_rows))

columns = ['filament_id', 'segmentation_rle']
df = pd.DataFrame(rows, columns=columns)
filename = 'submission_SMOKE_DO_NOT_SUBMIT.csv' if SMOKE else 'submission.csv'
CSV_PATH = OUT / filename
df.to_csv(CSV_PATH, index=False)
pd.DataFrame(completed).to_csv(OUT/'completed_test_images.csv', index=False)
print(CSV_PATH, 'rows:', len(df))

## 15. 保存内容の検査とKaggleでの提出

全行のRLEを復元し、ID、列、処理漏れを検査します。全て0本でもダミーを追加しません。完全に空の提出は学習や推論の失敗を疑い、大会のエッジケース検査も確認してください。

本実験が完了したら、KaggleでSave Versionし、OutputにCSV・重み・設定・ログが残っていることを確認して、大会の提出画面からCSVを選びます。ソースコードの提出・公開要件も最新ルールに従います。

In [ ]:
saved = pd.read_csv(CSV_PATH, keep_default_na=False)
assert list(saved.columns) == ['filament_id', 'segmentation_rle']
assert saved['filament_id'].is_unique
assert len(completed) == len(paths_to_run)
assert len({r['file_name'] for r in completed}) == len(paths_to_run)
known_stems = {p.stem for p in paths_to_run}
for row in saved.itertuples(index=False):
    assert row.filament_id.rsplit('_', 1)[0] in known_stems
    assert row.segmentation_rle and all(48 <= ord(c) <= 111 for c in row.segmentation_rle)
    m = mask_utils.decode(dict(size=[2048, 2048],
        counts=row.segmentation_rle.encode('ascii')))
    assert m.shape == (2048, 2048) and m.any()
    assert counts_string(m) == row.segmentation_rle
if not SMOKE:
    assert len(completed) == len(test_paths)
print('Checks passed:', CSV_PATH)
print('SMOKE output: DO NOT SUBMIT' if SMOKE else 'Full test set completed')

## 16. 数式を計算して復習

学習率や損失関数の意味を、最小の数値例で確認します。以下は大会の実測成績ではありません。

In [ ]:
import math
print('normalization:', 128/255)
print('neuron:', 0.5*0.2 - 0.3*0.8 + 0.1)
print('sigmoid(ln 4):', 1/(1 + math.exp(-math.log(4))))
print('BCE y=1,p=0.8:', -math.log(0.8))
tp, fp, fn = 6, 2, 4
print('IoU:', tp/(tp+fp+fn))
print('Dice:', 2*tp/(2*tp+fp+fn))
print('PQ example:', (0.8+0.7)/(2+0.5*1+0.5*1))
w, lr = 1.0, 0.1
gradient = 2*(w-3)
new_w = w-lr*gradient
print('gradient / new weight / loss:', gradient, new_w, (new_w-3)**2)

## 参考資料

- 大会：https://www.kaggle.com/competitions/filament-segmentation-2026
- 主催者の自己評価：https://www.kaggle.com/code/azimahmadzadeh/self-evaluation-notebook
- MAGFiLO：https://doi.org/10.1038/s41597-024-03876-y
- Ultralytics：https://docs.ultralytics.com/models/yolo26/
- YOLO分割ラベル：https://docs.ultralytics.com/datasets/segment/
- SAM 2 API：https://docs.ultralytics.com/models/sam-2/
- U-Net原資料：https://lmb.informatik.uni-freiburg.de/people/ronneber/u-net/
- SMP：https://smp.readthedocs.io/en/latest/models.html
- PyTorch：https://docs.pytorch.org/tutorials/beginner/basics/intro.html
- COCO RLE：https://github.com/cocodataset/cocoapi/blob/master/PythonAPI/pycocotools/mask.py

情報確認：2026年11月2日。KaggleのUIやライブラリのAPIが変わる場合があります。導入した版と実験設定を保存し、最新の公式資料と照合してください。